In [1]:
import os
import pandas as pd
import numpy as np
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

### Constants

In [2]:
str_dirname_output = './output'

### Output dir

In [3]:
try:
    os.mkdir(str_dirname_output)
except:
    pass

### Get data dictionary

In [4]:
str_filename = 'Data_Dictionary.xlsx'
str_local_path = f'./input/{str_filename}'
df = pd.read_excel(str_local_path)
df['feature'] = df['Column Name'].str.lower()
dict_map = {
    'Credit Vision (TU Accept)': '__tu',
    'Factor Trust (TU CVLINK)': '__tu',
    'LN': '__ln',    
}
df['suffix'] = df['Table'].map(dict_map)
df['feature'] = df['feature'] + df['suffix']
dict_map = dict(zip(df['feature'], df['Description']))

### Import features from old Gen XII

In [5]:
str_variant = 'noPTImodel7'
list_str_model = [
    '01_ad',
    '02_pricing_pd',
    '03_pricing_lgd',
]
list_cols_old = []
for str_model in tqdm(list_str_model):
    str_filename = 'df_cols_in_model.csv'
    str_uri = f's3://20231010-gen-xii/{str_model}/02_model/{str_variant}/02_model/01_lambda_get_starting_feats/{str_filename}'
    list_cols_tmp = list(pd.read_csv(str_uri)['feature'])
    list_cols_old += list_cols_tmp
list_cols_old = list(dict.fromkeys(list_cols_old))
print(f'There are {len(list_cols_old)} columns in the old Gen XII')

100%|██████████| 3/3 [00:00<00:00,  6.41it/s]

There are 519 columns in the old Gen XII


### Save the LN features

In [6]:
list_cols_old_ln = [col for col in list_cols_old if '__ln' in col.lower()]

# make df
df = pd.DataFrame({'feature': list_cols_old_ln})

# save
str_filename = 'df_ln.csv'
str_uri = f's3://20231010-gen-xii/ad_hoc/05_check_features/{str_filename}'
df.to_csv(str_uri, index=False)

# show
print(f'There are {len(list_cols_old_ln)} LN columns in the old Gen XII')
# for a, col in enumerate(list_cols_old_ln):
#     print(f'{a+1} - {col}')

There are 75 LN columns in the old Gen XII


### Import features from new Gen XII

In [7]:
str_variant = 'noPTImodel10'
list_str_model = [
    '01_ad',
    '02_pricing_pd',
    '03_pricing_lgd',
]
list_cols_new = []
for str_model in tqdm(list_str_model):
    str_filename = 'df_cols_in_model.csv'
    str_uri = f's3://20231010-gen-xii/{str_model}/02_model/{str_variant}/02_model/01_lambda_get_starting_feats/{str_filename}'
    list_cols_tmp = list(pd.read_csv(str_uri)['feature'])
    list_cols_new += list_cols_tmp
list_cols_new = list(dict.fromkeys(list_cols_new))
print(f'There are {len(list_cols_new)} columns in the new Gen XII')

100%|██████████| 3/3 [00:00<00:00, 16.69it/s]

There are 323 columns in the new Gen XII


### Map descriptions

In [8]:
list_dict_row = []
for a, col in enumerate(list_cols_new):
    # get description
    try:
        str_description = dict_map[col]
    except KeyError:
        str_description = 'Not in dictionary'
    print(f'{a+1} - {col}: {str_description}')
    dict_row = {
        'feature': col,
        'description': str_description,
    }
    list_dict_row.append(dict_row)
df = pd.DataFrame(list_dict_row)

# save
str_filename = 'df_allcols.csv'
str_local_path = f'{str_dirname_output}/{str_filename}'
df.to_csv(str_local_path, index=False)

1 - bankruptcycount24month__ln: Total unique bankruptcy case filings on file showing the subject as debtor in the last 24 months
2 - inquiryshortterm12month__ln: Indicates subject has one or more LexisNexis personal finance credit inquiries on file in the last 12 months (multiple personal finance inquiries within a 24-hour period are counted as one inquiry)
3 - re01s__tu: Number of revolving trades
4 - bankruptcystatus__ln: Indicates status of most recent bankruptcy filing on file showing the subject as debtor
5 - bankruptcytimenewest__ln: Time (in months) since most recent bankruptcy case filing on file showing the subject as debtor
6 - fltgrossmonthly__income_sum: Not in dictionary
7 - linka006__tu: Number of total inquiries in the last 3 years.  
These include checking, credit issuance (credit card, secured card, line of credit, revolving credit, and other credit account originations) , auto, payday, utility, and other credit (installment loan, secured loan, short-term installment l

### Identify features in new Gen XII not in old Gen XII

In [9]:
list_cols_additional = [col for col in list_cols_new if col not in list_cols_old]
print(f'There are {len(list_cols_additional)} columns in the new Gen XII that are not in the old Gen XII')
list_dict_row = []
for a, col in enumerate(list_cols_additional):
    # get description
    try:
        str_description = dict_map[col]
    except KeyError:
        str_description = 'Not in dictionary'
    print(f'{a+1} - {col}: {str_description}')
    dict_row = {
        'feature': col,
        'description': str_description,
    }
    list_dict_row.append(dict_row)
df = pd.DataFrame(list_dict_row)

# save
str_filename = 'df_newcols.csv'
str_local_path = f'{str_dirname_output}/{str_filename}'
df.to_csv(str_local_path, index=False)

There are 68 columns in the new Gen XII that are not in the old Gen XII
1 - at101b__tu: Total balance of all trades verified in past 12 months (excluding mortgage and home equity)
2 - g304s__tu: Worst rating on installment trades in past 12 months
3 - linkf111__tu: Number of lenders / locations with inquiries within last 6 months
4 - st32s__tu: Maximum balance owed on open student loan trades verified in past 12 months
5 - hi21s__tu: Months since most recent home equity loan trade opened
6 - at57s__tu: Total past due amount of open trades verified in past 12 months
7 - g200s__tu: Percentage of contractually liable debt
8 - mt21s__tu: Months since most recent mortgage trade opened
9 - linka039__tu: Number of days since first (oldest) check order
10 - g418s__tu: Months since most recent auto inquiry
11 - rvdex02__tu: Most recent quarter year-over-year revolving spend index
12 - rle904__tu: Amount of over-payments on Real-Estate accounts in the past 6 months
13 - rp01s__tu: Number of repo

### Show the LN features

In [10]:
list_cols_new_ln = [col for col in list_cols_additional if '__ln' in col]
print(f'There are {len(list_cols_new_ln)} LN columns in the new Gen XII that were not in the old Gen XII')
for a, col in enumerate(list_cols_new_ln):
    print(f'{a+1} - {col}')

There are 0 LN columns in the new Gen XII that were not in the old Gen XII
